# 04 · Functions

Turn a sequence of operations into a named, reusable piece of behavior. This chapter develops *4_functions.pdf* (pages 1–48), using noninteractive examples that work in a fresh kernel.

**Before you start:** know conditions, loops, and basic containers. Run cells in order. Expected errors are caught, and exercise cells are safe placeholders. Complete answers are in `../solutions/04_functions_solutions.ipynb`.


## What you will be able to do

- Define functions, return results, and distinguish printing from returning.
- Trace local, enclosing, global, and built-in name lookup.
- Explain object sharing, defaults, and the mutable-default trap.
- Use positional and keyword calls, keyword-only parameters, `*args`, and `**kwargs`.
- Document a clear interface and use functions as ordinary objects.

A **parameter** is a name in a function definition. An **argument** is a value supplied in a call. Keep those two roles distinct when reading signatures.


## 1. Define behavior and return a result

`def` creates a function object and binds it to a name. Its indented body runs when the function is called. `return expression` passes a value back to the caller and ends that call. The caller can store the result, pass it elsewhere, or use it in a larger expression.


In [ ]:
def rectangle_area(width, height):
    """Return the area of a rectangle with the supplied dimensions."""
    return width * height

area = rectangle_area(3, 4)
print(area)  # 12
print(rectangle_area(2, 5) + rectangle_area(1, 3))  # 13


### Every function returns something

Without an explicit return value, a function returns `None`. A bare `return` also returns `None`. Printing writes a display message; it does not make that message the function's result. Interactive displays often suppress a `None` result, but `print(None)` displays it explicitly.

Return several related values as a tuple and document the order. Callers can unpack that tuple. For a record with many named fields, a dictionary or a named record type can communicate the structure more clearly.


In [ ]:
def announce(message):
    print(message)


def positive_or_none(number):
    if number <= 0:
        return
    return number


def bounds(values):
    """Return (minimum, maximum); values must be nonempty."""
    return min(values), max(values)

result = announce("Hello")  # Prints Hello.
print(result is None, positive_or_none(0) is None)  # True True
low, high = bounds([4, 1, 8])
print(low, high, bounds([5]))  # 1 8 (5, 5)


## 2. Scope: where names are found

Each function call has local names, including its parameters. Assignment in a function normally binds a **local** name. Reusing a global name locally does not overwrite the global binding.

`locals()` can inspect local names; `globals()` exposes the module's global namespace. These are useful for inspection, but passing values through parameters and return values usually makes dependencies clearer than consulting namespace dictionaries.


In [ ]:
quantity = 2

def inspect_scope(increment):
    quantity = 41
    extra = 5
    snapshot = {name: value for name, value in locals().items()}
    print(snapshot)  # {'increment': 3, 'quantity': 41, 'extra': 5}
    print(globals()["quantity"])  # 2
    return quantity + increment + extra

print(inspect_scope(3))  # 49
print(quantity)  # 2 — the global name did not change


### LEGB: local → enclosing function → global → built-ins

For ordinary free-name lookup inside a function, Python searches local names, enclosing function scopes, the module's globals, and built-in names. A name absent from all these places raises `NameError`.

An assignment anywhere in a function makes that name local throughout that function unless declared `global` or `nonlocal`. Reading such a local before it receives a value raises `UnboundLocalError`, a subclass of `NameError`.


In [ ]:
location = "global"

def outer():
    location = "enclosing"
    def inner():
        local_value = "local"
        return local_value, location, len([1, 2])
    return inner()

print(outer())  # ('local', 'enclosing', 2) — len is a built-in

def broken_increment():
    quantity = quantity + 1  # The assignment makes quantity local here.
    return quantity

try:
    broken_increment()
except UnboundLocalError:
    print("Expected UnboundLocalError: local quantity was read before assignment.")
try:
    print(a_name_not_defined_in_this_chapter)
except NameError:
    print("Expected NameError: no matching binding was found.")


### Blocks and scope boundaries

`if`, `for`, `while`, and `with` do not introduce a new local scope. Assignments inside them belong to the surrounding function or module. A loop variable remains bound after a nonempty ordinary `for` loop.

The statement “only functions create scopes” is incomplete: modules, classes, and comprehensions have scope rules too. In Python 3, a comprehension's loop variable does not leak into the surrounding scope. Class bodies form namespaces and have special lookup rules; ordinary methods do not capture a class namespace as an enclosing function scope. LEGB is a useful model for functions, not a complete description of every Python construct.


In [ ]:
def block_example():
    if True:
        description = "created inside an if"
    for position in range(3):
        latest = position
    squares = [item ** 2 for item in range(3)]
    print(description, latest, position)  # created inside an if 2 2
    print(squares)  # [0, 1, 4]
    print("item" in locals())  # False — comprehension variable stays local to it

block_example()


### Arguments use object sharing

A parameter is a new local name bound to the argument's object. Mutating a shared mutable object can be observed by the caller. Rebinding the parameter to a different object does not rebind the caller's name. This behavior is often called **call by sharing** or passing object references; it is not a mechanism for reassigning the caller's variables.

An immutable value cannot be changed in place. To supply a new integer or string to the caller, return a new value and let the caller assign it.


In [ ]:
def mutate_and_rebind(values):
    values.append("shared change")
    values = ["local replacement"]
    return values

original = ["start"]
replacement = mutate_and_rebind(original)
print(original)     # ['start', 'shared change']
print(replacement)  # ['local replacement']

def increment(number):
    number += 1
    return number

counter = 10
print(increment(counter), counter)  # 11 10
counter = increment(counter)
print(counter)  # 11


### Exercise 04.1 · Clean a reading and return two values

Define `parse_reading(name, raw_value)` to return `(clean_name, numeric_value)`. Strip surrounding whitespace from the name and title-case it; convert the value with `float`. Return the tuple rather than printing it.

Expected: `parse_reading("  air TEMPERATURE ", " 21.5 ") == ("Air Temperature", 21.5)`. Check a negative reading and zero. Invalid numeric text should raise `ValueError`; catch that expected error in your check. Unpack a returned tuple into two variables.


In [ ]:
# Exercise 04.1
# TODO: Define parse_reading(name, raw_value), including a short docstring.
# TODO: Check normal, negative, zero, and invalid numeric inputs.


## 3. Default values simplify common calls

A default lets the caller omit an argument. In a conventional positional-or-keyword parameter list, required parameters come before parameters with defaults. Defaults should express sensible common behavior; parameters expose the settings a caller may want to change.

The source's yes/no prompt can be studied without `input()`: the following function reads a supplied iterable of responses. This makes it deterministic and prevents a notebook from waiting for keyboard input. It accepts at most `retries` responses and returns `False` if none is a recognized answer.


In [ ]:
def ask_yn(responses, retries=4, complaint="Enter Y/N!"):
    """Return the first Y/N decision within retries; otherwise return False."""
    if retries < 0:
        raise ValueError("retries must be nonnegative")
    responses = iter(responses)
    for _ in range(retries):
        try:
            answer = next(responses).strip().upper()
        except StopIteration:
            break
        if answer == "Y":
            return True
        if answer == "N":
            return False
        print(complaint)
    return False

print(ask_yn(["Y"]))  # True
print(ask_yn(["?", "N"], complaint="Use Y or N."))  # complaint, then False
print(ask_yn(["?", "Y"], retries=1))  # complaint, then False
print(ask_yn([], retries=0))  # False


### Defaults are evaluated when the function is defined

A default expression is evaluated **once per execution of the definition**, not on every call. Changing the original name later does not change the stored default. A mutable default can therefore accumulate state across calls unintentionally.

Use `None` as a sentinel and create a new list inside the function when omitted. `if items is None` is intentional: `if not items` would also replace a caller's explicitly supplied empty list.


In [ ]:
base_rate = 2

def scale(value, factor=base_rate):
    return value * factor

base_rate = 10
print(scale(3), scale(3, base_rate))  # 6 30

def unsafe_collect(value, items=[]):
    items.append(value)
    return items

first = unsafe_collect("a")
second = unsafe_collect("b")
print(first, second, first is second)  # ['a', 'b'] ['a', 'b'] True

def safe_collect(value, items=None):
    if items is None:
        items = []
    items.append(value)
    return items

print(safe_collect("a"), safe_collect("b"))  # ['a'] ['b']
provided = []
print(safe_collect("c", provided) is provided, provided)  # True ['c']


### Exercise 04.2 · Add labels without accidental sharing

Define `add_label(label, labels=None)` to append `label.strip()` to a list and return that list. When `labels` is omitted or `None`, create a new list. When a list is explicitly supplied, mutate and return that exact list.

Expected: separate calls `add_label(" A ")` and `add_label("B")` produce `['A']` and `['B']` as different objects. If `existing = []`, then `add_label(" C ", existing) is existing` and `existing == ['C']`. Empty label text becomes `''`; no filtering is required. Explain why the default should not be `[]`.


In [ ]:
# Exercise 04.2
# TODO: Define add_label with a safe default and document its mutation behavior.
# TODO: Test separate omitted defaults, an explicit empty list, and an empty label.


## 4. How arguments bind to parameters

Ordinary parameters can be supplied by position or by keyword. Keywords make settings readable and can appear in any order. A parameter with a default is not automatically keyword-only: it may still be passed positionally unless the signature says otherwise.

Built-ins have different signatures. For example, `enumerate(values, start=1)` accepts a keyword for `start`, while `range` takes positional arguments. Inspect the function's documented signature rather than assuming every named concept is a valid keyword.


In [ ]:
def describe_device(voltage, state="ready", action="run", device_type="sensor"):
    return f"{device_type}: {voltage} V, {state}, {action}"

print(describe_device(5))
print(describe_device(voltage=5))
print(describe_device(action="measure", voltage=12))
print(describe_device(12, "standby", "wait"))
print(describe_device(3.3, state="active", device_type="probe"))
# sensor: 5 V, ready, run
# sensor: 5 V, ready, run
# sensor: 12 V, ready, measure
# sensor: 12 V, standby, wait
# probe: 3.3 V, active, run


### Invalid calls: recognize the failure

A required parameter cannot be omitted, a parameter cannot receive two values, and an unknown keyword is rejected unless the function accepts extra keywords. An ordinary positional argument cannot follow a keyword argument in call syntax.

The first three mistakes below raise `TypeError` when the call happens. The last is invalid **syntax**, so the example passes a small string to `compile` and catches `SyntaxError`; the notebook itself remains executable.


In [ ]:
invalid_calls = [
    ("missing required argument", lambda: describe_device()),
    ("duplicate voltage", lambda: describe_device(5, voltage=12)),
    ("unknown keyword", lambda: describe_device(5, owner="Ada")),
]
for reason, call in invalid_calls:
    try:
        call()
    except TypeError:
        print(f"Expected TypeError: {reason}.")
try:
    compile("describe_device(voltage=5, 'ready')", "<call example>", "exec")
except SyntaxError:
    print("Expected SyntaxError: ordinary positional argument follows keyword.")


## 5. Variable numbers of positional arguments

In a definition, `*numbers` collects extra positional arguments into a tuple. The name `args` is conventional, not mandatory. Parameters after `*numbers` are **keyword-only**. A bare `*` also marks a keyword-only boundary, for example `def report(values, *, verbose=False): ...`.

In a call, `*iterable` performs the opposite job: it unpacks items into positional arguments. The product below uses `1` as the multiplicative identity, so an empty product returns the scale.


In [ ]:
def product(*numbers, scale=1):
    """Return scale multiplied by every supplied number."""
    result = scale
    for number in numbers:
        result *= number
    return result

print(product(3, 5), product(3, 4, 2), product(3, 5, scale=10))  # 15 24 150
print(product(), product(scale=10))  # 1 10
primes_under_20 = [2, 3, 5, 7, 11, 13, 17, 19]
print(product(*primes_under_20))  # 9699690
print(product(3, 5, 10))  # 150: 10 is another number, not the scale parameter

def divide(numerator, *, denominator=1):
    return numerator / denominator

print(divide(12, denominator=3))  # 4.0
try:
    divide(12, 3)
except TypeError:
    print("Expected TypeError: denominator must be supplied by keyword.")


## 6. Variable numbers of keyword arguments

In a definition, `**metadata` collects unmatched keyword arguments into a dictionary. In a call, `**mapping` unpacks string keys and values into keyword arguments. A function's `**kwargs` is a new dictionary, but its values may still refer to shared mutable objects.

Current Python preserves the supplied keyword order. Choose explicit sorting when output order should be independent of the caller's insertion order. Duplicate parameter values are still errors even when they arrive through unpacking.


In [ ]:
def format_quote(quote, **speaker_info):
    """Return a quotation followed by metadata in caller-supplied order."""
    lines = [f"> {quote}", "-" * (len(quote) + 2)]
    for key, value in speaker_info.items():
        lines.append(f"{key}: {value}")
    return "\n".join(lines)

print(format_quote("Practice makes progress.", author="Anonymous", topic="learning"))
info = {"author": "Shakespeare", "sonnet": 18, "line": 1}
print(format_quote("Shall I compare thee to a summer's day?", **info))
try:
    describe_device(5, **{"voltage": 12})
except TypeError:
    print("Expected TypeError: unpacking also cannot bind voltage twice.")
try:
    format_quote("Example", **{1: "not a string key"})
except TypeError:
    print("Expected TypeError: unpacked keyword names must be strings.")


### Read a combined signature

In `combine(a, b, c=1, *extra, e=1, **metadata)`, `a` and `b` are required positional-or-keyword parameters, `c` has a default, `extra` collects remaining positional arguments, `e` is keyword-only, and `metadata` collects unmatched keywords.

Modern Python also supports `/` to mark positional-only parameters, as in `def identity(value, /): ...`. Use these boundaries when they make an interface clearer, rather than adding every parameter style to every function.


In [ ]:
def combine(a, b, c=1, *extra, e=1, **metadata):
    return {"required": (a, b), "c": c, "extra": extra, "e": e, "metadata": metadata}

print(combine(10, 20))
# {'required': (10, 20), 'c': 1, 'extra': (), 'e': 1, 'metadata': {}}
print(combine(10, 20, 30, 40, 50, e=2, unit="kg"))
# {'required': (10, 20), 'c': 30, 'extra': (40, 50), 'e': 2, 'metadata': {'unit': 'kg'}}
arguments = [10, 20, 30, 40]
options = {"e": 2, "unit": "kg"}
print(combine(*arguments, **options))

def identity(value, /):
    return value

print(identity("ok"))
try:
    identity(value="ok")
except TypeError:
    print("Expected TypeError: value is positional-only.")


### Exercise 04.3 · A flexible total

Define `adjusted_total(*amounts, multiplier=1, offset=0)` to return the sum of the amounts multiplied by `multiplier`, plus `offset`. Both settings must be keyword-only.

Expected: `adjusted_total(2, 3) == 5`, `adjusted_total(2, 3, multiplier=2, offset=1) == 11`, and `adjusted_total(offset=7) == 7`. Unpack `[1, 2, 3]` into a call and expect `6`. Check a negative amount and a fractional multiplier. Explain why `adjusted_total(2, 3, 4)` treats `4` as an amount.


In [ ]:
# Exercise 04.3
# TODO: Define adjusted_total with *amounts and keyword-only settings.
# TODO: Check defaults, unpacking, empty input, and negative/fractional values.


## 7. Formatting is another argument-binding example

`str.format(*args, **kwargs)` accepts both forms of arguments. `{0}` selects positional argument zero; `{name}` selects a keyword. Argument unpacking works here just as it does with your functions. F-strings are often the clearest choice when the values are already in local variables.

A format string can ignore extra keyword entries. Expanding all of `locals()` may seem convenient for debugging, but an explicit mapping or f-string makes a function's dependencies easier to read and control.


In [ ]:
print("First count to {0}".format(3))  # First count to 3
print("{0}{b}{1}{a}{0}{2}".format(5, 8, 9, a="z", b="x"))  # 5x8z59
arguments = (5, 8, 9)
keywords = {"a": "z", "b": "x"}
print("{0}{b}{1}{a}{0}{2}".format(*arguments, **keywords))  # 5x8z59
triangle = {"x": 3, "y": 4, "z": 5}
print("{z}^2 = {x}^2 + {y}^2".format(**triangle))  # 5^2 = 3^2 + 4^2
x, y, z = 3, 4, 5
print(f"{z}^2 = {x}^2 + {y}^2")  # 5^2 = 3^2 + 4^2


## 8. Document a useful contract

A function's first statement may be a string literal: its **docstring**. Start with a short summary, then explain arguments, return values, mutation, and errors when those are relevant. The docstring is available through `__doc__` and help tools.

Use four spaces for indentation, `snake_case` for function and variable names, spaces around operators and after commas, and blank lines between top-level functions. Choose small functions with clear responsibilities. Comments explain nonobvious decisions; avoid merely translating each line into English. A module docstring can explain a standalone file's purpose.


In [ ]:
def mean(values):
    """Return the arithmetic mean of a nonempty sequence of numbers.

    The input sequence is not modified.
    Raise ValueError if values is empty.
    """
    if not values:
        raise ValueError("mean requires at least one value")
    return sum(values) / len(values)

print(mean([2, 4, 9]))  # 5.0
print(mean.__name__)   # mean
print(mean.__doc__)
try:
    mean([])
except ValueError as error:
    print(f"Expected ValueError: {error}")


### Exercise 04.4 · Format an event with metadata

Define `format_event(event, **details)` to return one string: the event name on the first line, followed by `key: value` lines in **alphabetical key order**. Do not print inside the function. Add a docstring describing the return format.

Expected: `format_event("Saved", user="Ada", count=3) == "Saved\ncount: 3\nuser: Ada"`. With no details, return only `"Saved"`. Unpack a dictionary in a call and show that changing its insertion order does not change the result. The caller's dictionary must remain unchanged.


In [ ]:
# Exercise 04.4
# TODO: Define format_event and build its lines in sorted key order.
# TODO: Test empty metadata, **dictionary unpacking, and unchanged input.


## 9. Functions are objects

A function can be assigned to another name, placed in a container, passed as an argument, and returned from another function. The alias points to the same function object; calling either name performs the same operation.

Functions have attributes such as `__name__` and `__doc__`. You can attach custom attributes, but explicit parameters or a suitable data structure usually communicate state more clearly. We inspect identity with `is` rather than printing nondeterministic memory addresses.


In [ ]:
def echo(value):
    """Return the supplied object unchanged."""
    return value

alias = echo
print(type(echo).__name__, alias is echo, isinstance(echo, object))  # function True True
print(alias("hello"), echo.__name__)  # hello echo

def apply_twice(function, value):
    return function(function(value))

print(apply_twice(str.upper, "hello"))  # HELLO

def make_multiplier(factor):
    def multiply(value):
        return value * factor
    return multiply

triple = make_multiplier(3)
print(triple(7), make_multiplier(2)(7))  # 21 14
echo.category = "demonstration"
print(echo.category)  # demonstration


### Enclosing state and explicit rebinding

A returned inner function can keep access to an enclosing function's variables; this is a **closure**. Reading an enclosing name needs no declaration. To rebind it, use `nonlocal`. A `global` declaration instead directs assignment to the module's namespace.

Prefer explicit inputs and return values for ordinary calculations. Use stateful closures deliberately, and document that successive calls can behave differently. The next example keeps each counter's state independent.


In [ ]:
def make_counter(start=0):
    count = start
    def next_count():
        nonlocal count
        count += 1
        return count
    return next_count

counter_a = make_counter()
counter_b = make_counter(10)
print(counter_a(), counter_a(), counter_b(), counter_a())  # 1 2 11 3


### Exercise 04.5 · Supply the transformation

Define `transform_values(values, transform)` to return a new list containing `transform(value)` for each input value. Do not change the input list. Then define `make_offset(offset)` to return a function that adds `offset` to its argument.

Expected: with `values = [1, 2, 3]`, `transform_values(values, make_offset(10)) == [11, 12, 13]` and `values` remains `[1, 2, 3]`. Try `str` as a transformation and test an empty list. Create two offset functions and verify they retain different offsets.


In [ ]:
# Exercise 04.5
# TODO: Define transform_values and a function-returning make_offset.
# TODO: Check unchanged input, empty input, built-in callbacks, and independent closures.


## 10. Put the interface first

Before writing a function body, decide what inputs it accepts, what result it returns, what it changes, and which cases count as errors. Small helpers can make these decisions visible. The following normalizer copies its output and uses a keyword-only setting so the call explains what is changing.


In [ ]:
def normalize_names(names, *, uppercase=False):
    """Return stripped names in title case, or uppercase when requested.

    Empty names remain empty strings. The input iterable is not modified.
    """
    normalized = [name.strip() for name in names]
    if uppercase:
        return [name.upper() for name in normalized]
    return [name.title() for name in normalized]

raw_names = [" ADA ", "bo", ""]
print(normalize_names(raw_names))  # ['Ada', 'Bo', '']
print(normalize_names(raw_names, uppercase=True))  # ['ADA', 'BO', '']
print(raw_names)  # [' ADA ', 'bo', '']


### Exercise 04.6 · Summarize scores with a clear contract

Define `summarize_scores(scores, *, pass_mark=50)` for a sequence of numeric scores. Return a dictionary with `count`, `mean`, and `passed`. `passed` counts scores greater than or equal to `pass_mark`. Do not modify the input.

For empty input, return `{'count': 0, 'mean': None, 'passed': 0}`. For `[40, 50, 90]`, expect `{'count': 3, 'mean': 60.0, 'passed': 2}`; with `pass_mark=80`, only one score passes. Document the empty-input rule. Verify the threshold boundary, a singleton, and that passing `pass_mark` positionally raises a caught `TypeError`.


In [ ]:
# Exercise 04.6
# TODO: Define summarize_scores with a docstring and keyword-only pass_mark.
# TODO: Test ordinary, empty, singleton, boundary, and invalid-call cases.


## Check your understanding

- Why does a function that prints a message usually return `None`?
- Which of mutation and rebinding can affect a caller's list?
- When are default expressions evaluated, and why is `None` a useful sentinel?
- How do `*` and `**` differ between a definition and a call?
- How can two functions returned by the same factory remember different values?

Review [the six complete solutions](../solutions/04_functions_solutions.ipynb), or run [the standalone examples](../examples/04_functions.py). The next chapter introduces [objects and classes](05_oop.ipynb).
